# NSGABlack 端到端陪跑：从一个真实问题开始（投资组合只是例子）

你现在手上的任务，多半不是“教科书算法题”，而更像这种：

- 目标不止一个（效果、成本、风险、鲁棒性……）
- 约束一堆，而且会变（硬约束、软偏好、业务规则、性能限制）
- 你想快速落实新点子：今天加一个偏好，明天换一种策略，后天要统一输出口径

这份 Notebook 就当我坐在你旁边“陪你跑一遍”：我们不讲概念展示，直接把一个完整流程从零跑通。

你要记住的只有四个角色（也是你以后最常换的积木）：

- `Problem`：只负责“怎么算目标”
- `RepresentationPipeline`：只负责“解怎么表示 + 怎么修到可行”
- `BiasModule`：只负责“软偏好/经验怎么加进去”
- `Suite/Plugin`：只负责“怎么装配算法 + 怎么统一输出口径”

最后你会拿到两份可复用、可追踪的输出：

- `runs/notebook_portfolio/portfolio_moead.csv`（逐步记录）
- `runs/notebook_portfolio/portfolio_moead.summary.json`（一次运行口径总结）

先别担心“投资组合我不做怎么办”：你只要把 `evaluate(x)` 换成自己的业务评估，其它结构几乎不用动。


## 0) 环境准备：保证能 import（你一定会遇到这一步）

这一节看起来不像“算法”，但它肯定是你第一次使用时最容易卡的地方：

- `python -m nsgablack ...` 突然给你一句 `No module named nsgablack`

这不是你写错了，而是 Python 没把“仓库根目录”当成包路径。

你有两个选择：

1) 推荐：在仓库根目录执行一次 `pip install -e .`（后面所有终端/Notebook 都省心）
2) 临时：设置 `PYTHONPATH=..`（或者 Notebook 里加 sys.path）

下面这个小函数做的事情就是：从当前目录往上找，直到能 import 成功为止。


In [ ]:
import os
import sys
from pathlib import Path

import numpy as np


def ensure_importable() -> Path:
    """确保可以 import nsgablack，并返回仓库根目录（也就是 package 目录）。"""
    try:
        import nsgablack  # noqa: F401
        return Path.cwd().resolve()
    except Exception:
        pass

    start = Path.cwd().resolve()
    for p in [start, *start.parents]:
        # 这个仓库布局：repo root 目录本身就是 package（含 __init__.py + pyproject.toml）
        if (p / "pyproject.toml").exists() and (p / "__init__.py").exists():
            repo_root = p
            repo_parent = str(repo_root.parent)
            if repo_parent not in sys.path:
                sys.path.insert(0, repo_parent)
            return repo_root

    raise RuntimeError("未找到仓库根目录：请在 repo 内运行，或先执行 pip install -e .")


repo_root = ensure_importable()
print("repo_root:", repo_root)

from nsgablack.core.base import BlackBoxProblem
from nsgablack.core.composable_solver import ComposableSolver
from nsgablack.adapters import MOEADConfig
from nsgablack.representation import RepresentationPipeline
from nsgablack.representation.continuous import GaussianMutation, UniformInitializer
from nsgablack.utils.suites import attach_benchmark_harness, attach_moead
from nsgablack.bias import BiasModule
from nsgablack.bias.domain import CallableBias


## 1) 定义问题（Problem）：先把“目标函数”写稳

你现在只做一件事：把业务目标写成一个稳定的 `evaluate(x)`。

这时候你很可能会问：

- “那我的约束呢？不满足就给大惩罚不行吗？”
- “既然都要计算，我在里面顺便打印日志/记录统计不行吗？”

都能跑，但它会在你开始“变化”时反咬你：换算法、加并行、改实验输出口径都要拆开重写。

所以我们这里很坚决：

- `Problem` 就几行，只发生一次“输入 x -> 目标 f”
- 硬约束：交给 `RepresentationPipeline.repair`
- 软偏好：交给 `BiasModule`
- 记录/统计：交给 `Plugin`

检查点：
- 这一节结束后，你应该可以对任意一个随机 `x` 调用 `problem.evaluate(x)`，它不依赖任何算法。


In [ ]:
class PortfolioProblem(BlackBoxProblem):
    def __init__(self, *, n_assets: int = 10, cap: float = 0.3, seed: int = 0):
        self.n_assets = int(n_assets)
        self.cap = float(cap)
        rng = np.random.default_rng(seed)

        # 构造半正定协方差矩阵（risk）
        A = rng.normal(size=(self.n_assets, self.n_assets))
        cov = A.T @ A
        cov = cov / max(1e-12, float(np.max(np.abs(cov))))
        self.cov = cov

        # 期望收益（return）
        self.mu = rng.uniform(0.01, 0.20, size=(self.n_assets,))

        # 注意：这里 bounds 只是“变量范围描述”，真正的硬约束我们仍然在 repair 里保证。
        bounds = {f"w{i}": [0.0, self.cap] for i in range(self.n_assets)}
        super().__init__(
            name="portfolio",
            dimension=self.n_assets,
            bounds=bounds,
            objectives=["min_risk", "min_neg_return"],
        )
        # 让变量名与 bounds key 对齐（避免 x0/x1 与 w0/w1 混淆）
        self.variables = list(bounds.keys())

    def evaluate(self, x):
        w = np.asarray(x, dtype=float).reshape(-1)
        # 目标1：风险（方差）
        risk = float(w.T @ self.cov @ w)
        # 目标2：最大化收益 => 最小化负收益
        neg_ret = float(-(w @ self.mu))
        return [risk, neg_ret]


problem = PortfolioProblem(n_assets=10, cap=0.3, seed=42)
print("mu:", np.round(problem.mu, 3))
print("cov shape:", problem.cov.shape)


## 2) 表示与修复（RepresentationPipeline）：把硬约束搬出 Problem

这一节的目标是：让你以后不用再在 `evaluate` 里刷一大段惩罚项。

你往往会遇到这种现实：
- 解空间很大，不可行的解也很多
- 算法一半时间都用来评估“明明一看就不合法”的解

更稳的方式是：
- `mutate` 负责生成邻域/扰动
- `repair` 负责把扰动后的解拉回可行域

在这个例子里，我们有两个经典硬约束：
- `0 <= w_i <= cap`
- `sum(w) = 1`

下面这个 repair 就是一个“很常用，很容易改”的版本：先 clip，再 normalize，迭代几次直到稳定。

检查点：
- 你随手给一个“乱七八糟”的向量，`repair` 后应该满足边界与求和。


In [ ]:
class CappedSimplexRepair:
    def __init__(self, cap: float = 0.3, iters: int = 12):
        self.cap = float(cap)
        self.iters = int(iters)

    def repair(self, x, context=None):
        w = np.asarray(x, dtype=float).reshape(-1)
        w = np.clip(w, 0.0, self.cap)

        for _ in range(max(1, self.iters)):
            s = float(np.sum(w))
            if s <= 1e-12:
                w[:] = 1.0 / float(w.size)
            else:
                w = w / s
            w = np.clip(w, 0.0, self.cap)

        # 收尾：再做一次 clip + normalize，避免最后一步破坏 cap
        w = np.clip(w, 0.0, self.cap)
        s = float(np.sum(w))
        if s > 1e-12:
            w = w / s
        w = np.clip(w, 0.0, self.cap)
        s = float(np.sum(w))
        if s > 1e-12:
            w = w / s
        return w


pipeline = RepresentationPipeline(
    initializer=UniformInitializer(low=0.0, high=problem.cap),
    mutator=GaussianMutation(sigma=0.05),
    repair=CappedSimplexRepair(cap=problem.cap),
)

x0 = pipeline.init(problem, {"generation": 0})
print("sum(w)=", float(np.sum(x0)))
print("max(w)=", float(np.max(x0)))
print("is_valid=", bool(problem.is_valid(x0)))


## 3) 软偏好（Bias）：让“你想要的解”更容易出现

这时候你往往会发现：硬约束通了，但“解长得不像你想要的”。

比如：
- 你希望更分散（不要过度集中在少数变量）
- 你希望更稀疏/更可解释
- 你希望某些区域“多探探”，某些区域“少碰碰”

这些不一定是硬约束，你也不想写死。所以我们用 Bias：
- 你只是在目标上叠一层 penalty/reward
- 可以开关，可以调权重，可以写很多个并存

这里我们用一个最简单的偏好：惩罚集中度（`sum(w^2)` 越大越集中）。

检查点：
- 你可以先把 `weight=0.0` 当成“关闭 Bias”，确认主流程不被耦合住。
- 然后再一点点调重，看它怎么影响解的风格。


In [ ]:
def concentration_penalty(w: np.ndarray) -> float:
    w = np.asarray(w, dtype=float).reshape(-1)
    return float(np.sum(w * w))


bias = BiasModule()
bias.add(
    CallableBias(
        name="prefer_diversification",
        func=concentration_penalty,
        weight=0.10,
        mode="penalty",
    )
)

print(bias)


## 4) 组装求解器：把“算法选择”变成可替换的积木

这时候你下意识会想：“那我现在就开始写算法了吧”。

但你很快会发现两个很巨大的摩擦成本：

- 算法本身很长，而你其实只想“快速用起来”
- 你总会漏掉一些必配组件（日志、archive、统计口径、并行设置……）

所以这个框架的“权威路径”是：

- `ComposableSolver` 管装配，尽量不投放额外逻辑
- `Suite` 管“一键把必配装齐”（它负责降低漏配概率）
- `Plugin` 管统一输出口径（这会让你以后比较算法时不再“说不清”）

本例选用：`attach_moead` + `attach_benchmark_harness`。

小习惯（很值得养成）：
- 你可以先用 `catalog search` 搜一下名字，看看它的推荐来源和伙伴组件
- 然后直接用 suite 一键装好，不要从“一大堆类”里手工挑

关于随机性：
- 你可以不固定 seed（多跑几次更接近工程习惯）
- 但你一定要记录 seed/time/eval_count，否则结果就没法比较（BenchmarkHarnessPlugin 就是为了这个）


In [ ]:
out_dir = os.path.abspath(os.path.join(str(repo_root), "runs", "notebook_portfolio"))
os.makedirs(out_dir, exist_ok=True)

np.random.seed(123)

solver = ComposableSolver(
    problem=problem,
    representation_pipeline=pipeline,
    bias_module=bias,
)

# 给 MOEA/D 一个更轻量的配置，Notebook 里跑起来更快
moead_cfg = MOEADConfig(population_size=60, neighborhood_size=15, batch_size=40, random_seed=123)
attach_moead(solver, config=moead_cfg, archive=True)

# 统一实验口径：每一步写一行 CSV，最后写 summary JSON
attach_benchmark_harness(
    solver,
    output_dir=out_dir,
    run_id="portfolio_moead",
    seed=123,
    overwrite=True,
)

solver.max_steps = 60
result = solver.run()

print("status:", result.get("status"))
print("steps:", result.get("steps"))
print("out_dir:", out_dir)


## 5) 看结果：先抓“口径”，再谈“性能”

跑完一次后，你大概率会想立刻调参，甚至想换算法。

但我建议你先做一件很“现实”的事：确认输出口径是完整的，否则你后面所有比较都是“说不清”的。

所以我们创建一个可信的节奏：

1) 先看 summary：seed/time/eval_count/best_score 都在不在
2) 再看 CSV 前几行：step 是不是递增，best_score 是不是在动
3) 如果是多目标：再看 Pareto archive（这是你真正的产物）

到这一步，你才开始“安心”地调参、换策略、加偏好。


## 6) 输出检查点：你应该看到什么

这一段我建议你边跑边对照：

- 如果 `.csv` 和 `.summary.json` 都生成了：说明“实验口径链路”是通的
- 如果 summary 里缺字段：优先检查是不是没挂 `attach_benchmark_harness`
- 如果 CSV 里 best_score 一直不变：先别急着怀疑算法，看看 repair/bias 是不是把问题搬得太难了

下面我们直接打印输出路径，然后读 summary 与 CSV 前几行。


In [ ]:
csv_path = os.path.join(out_dir, "portfolio_moead.csv")
summary_path = os.path.join(out_dir, "portfolio_moead.summary.json")
print("csv:", csv_path)
print("summary:", summary_path)


In [ ]:
import csv
import json

with open(summary_path, "r", encoding="utf-8") as f:
    summary = json.load(f)
print("summary:")
print(summary)

print("\nCSV 前 5 行：")
with open(csv_path, "r", encoding="utf-8") as f:
    reader = csv.reader(f)
    for i, row in enumerate(reader):
        print(row)
        if i >= 5:
            break


## 7) 多目标的核心产物：Pareto archive

如果你做的是多目标优化，你最后真正想要的不是“一个最优解”，而是一组可以互相 trade-off 的解。

`attach_moead` 默认会挂一个 archive 插件，求解过程中会不断维护当前的 Pareto 集。

下面我们把它拿出来：
- `pareto_solutions`：对应的决策变量（这里就是一组权重向量）
- `pareto_objectives`：对应的目标值

这一组输出很适合用作：
- 下一阶段（比如 VNS/局部搜索）“深挖”的起点
- 或者交给业务侧做二次筛选/可解释分析的输入


In [ ]:
pareto_X = getattr(solver, "pareto_solutions", None)
pareto_F = getattr(solver, "pareto_objectives", None)
pareto_V = getattr(solver, "pareto_violations", None)

if pareto_X is None:
    print("未检测到 Pareto archive（你是否在 attach_moead 里把 archive 关掉了？）")
else:
    X = np.asarray(pareto_X)
    F = np.asarray(pareto_F, dtype=float)
    V = None if pareto_V is None else np.asarray(pareto_V, dtype=float)
    print("pareto_solutions shape:", X.shape)
    print("pareto_objectives shape:", F.shape)
    if V is not None:
        print("max_violation:", float(np.max(V)))
    print("前 5 个目标值（risk, -return）：")
    print(F[:5])
